> **说明**：本 notebook 是 [langchain-ai/langgraph-101](https://github.com/langchain-ai/langgraph-101) 的**简体中文翻译版**，供 Google Colab / Jupyter 使用。
>
> - 原文仓库：<https://github.com/langchain-ai/langgraph-101>
> - 建议学习顺序：`101_langchain_langgraph` → `102_middleware` → `deep_agents` → `deep_agents_extra`
> - **代码单元格保持英文原样**（标识符、字符串、prompt 均未改动）；仅翻译 Markdown 与 `#` 注释。
> - 在 Colab 中请先运行 **Colab 一键环境** 单元格；图片与 `utils/` 已包含在本仓库中。


# Deep Agents Extra：深入内部机制

201 教你如何**使用** Deep Agents。本 notebook 教你它们如何**工作** —— 以及如何定制真正重要的部分。

**你将学到：**
- `create_deep_agent()` 本质上就是 `create_agent()` + 中间件
- 如何构建自己的文件系统接口（讲模式，而非完整实现）
- 如何用结构化任务定义替换默认的 `task()` 工具
- 如何把确定性的 `StateGraph` 工作流作为子智能体接入
- MCP 适配器如何让外部工具与原生工具可互换
- 如何用沙箱做代码执行
- 上下文管理失败时会发生什么 —— 以及如何修复

**前置条件：** 请先完成 201 Deep Agents notebook。


## 第 0 部分：环境准备


In [ ]:
# Colab 一键环境：安装依赖、在 Colab 中克隆本仓库（若缺少 utils），并配置 sys.path
# 本地完整 clone 时可跳过 pip/clone，但仍建议运行本单元格以确保路径正确
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Sucran/langgraph-101-zh.git"
COLAB_REPO_DIR = Path("/content/langgraph-101-zh")


def _in_colab() -> bool:
    if Path("/content").exists():
        return True
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if _in_colab():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--pre",
            "-U",
            "langchain",
            "langchain-anthropic",
            "langchain-openai",
            "python-dotenv",
            "deepagents",
            "tavily-python",
        ]
    )
    if not (COLAB_REPO_DIR / "utils" / "models.py").is_file():
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)]
        )


def find_project_root() -> Path:
    """Repo root containing utils/models.py (local clone or Colab clone)."""
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "utils" / "models.py").is_file():
            return p
    if COLAB_REPO_DIR.is_dir() and (COLAB_REPO_DIR / "utils" / "models.py").is_file():
        return COLAB_REPO_DIR
    return Path.cwd().resolve().parent.parent


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if not os.environ.get("ANTHROPIC_API_KEY") and not os.environ.get("OPENAI_API_KEY"):
    print(
        "请设置 API Key：Colab 左侧「密钥」→ ANTHROPIC_API_KEY（默认模型）或 OPENAI_API_KEY；"
        "也可执行：import os; os.environ['ANTHROPIC_API_KEY'] = '...'"
    )
if not os.environ.get("TAVILY_API_KEY"):
    print(
        "Deep Agents 需要 TAVILY_API_KEY：Colab「密钥」或 os.environ['TAVILY_API_KEY'] = '...'"
    )


In [ ]:
# 将项目根目录加入 Python 路径
import sys
from pathlib import Path


def find_project_root() -> Path:
    colab_repo = Path("/content/langgraph-101-zh")
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "utils" / "models.py").is_file():
            return p
    if colab_repo.is_dir() and (colab_repo / "utils" / "models.py").is_file():
        return colab_repo
    return Path.cwd().resolve().parent.parent


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
print(f"Model: {model.__class__.__name__}")


In [ ]:
# 本 notebook 的共享导入
from langchain_core.tools import tool
from langsmith import uuid7
from langgraph.checkpoint.memory import MemorySaver

from tavily import TavilyClient
tavily_client = TavilyClient()

@tool(parse_docstring=True)
def tavily_search(query: str) -> str:
    """Search the web for information on a given query.
    
    Args:
        query: Search query to execute.
    """
    results = tavily_client.search(query, max_results=3)
    return "\n\n".join(
        f"**{r['title']}**\n{r['url']}\n{r['content']}" 
        for r in results["results"]
    )

---
## 第 1 部分：拆解 `create_deep_agent`

`create_deep_agent()` 是围绕 `create_agent()` 的封装，它组装了一套特定的中间件栈。

它大致做了这些事（简化版）：

```python
def create_deep_agent(model, tools, system_prompt, ...):
    middleware = [
        TodoListMiddleware(),                    # Planning
        FilesystemMiddleware(backend=backend),   # File tools
        SubAgentMiddleware(subagents=...),       # task() tool
        SummarizationMiddleware(model, ...),     # Context management
        PatchToolCallsMiddleware(),              # Fix dangling tool calls
    ]
    return create_agent(model, tools=tools, middleware=middleware)
```

### 为什么这很重要

如果你能自己搭中间件栈，就可以：
- 替换任意部件（例如自己的文件系统、自己的 task 工具）
- 以不同顺序添加中间件
- 直接使用 `create_agent()`，无需 deep agents 依赖


In [ ]:
# 用 create_agent 从零构建同样的东西
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.middleware.summarization import create_summarization_middleware
from deepagents.middleware.patch_tool_calls import PatchToolCallsMiddleware
from deepagents.backends import StateBackend

# 这就是 create_deep_agent 的核心所做之事
hand_built_agent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt="You are a helpful research assistant.",
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        create_summarization_middleware(model, StateBackend),
        PatchToolCallsMiddleware(),
    ],
).with_config({"recursion_limit": 1000})

print("Agent built with create_agent + middleware")
print(f"Type: {type(hand_built_agent).__name__}")

In [ ]:
# 测试 —— 行为与 create_deep_agent 完全一致
config = {"configurable": {"thread_id": str(uuid7())}}

result = hand_built_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file called /notes.md with 'Built from scratch!' then read it back."}]
}, config=config)

print(result["messages"][-1].content)
print("\nFiles in state:", list(result.get("files", {}).keys()))

### 中间件 hooks 一览

每个中间件可实现这些 hooks：

| Hook | 何时运行 | 常见用途 |
|------|-------------|------------|
| `before_agent(state, runtime)` | 开始时一次 | 加载 skills、记忆 |
| `wrap_model_call(request, handler)` | 每次 LLM 调用 | 注入系统提示文本 |
| `wrap_tool_call(request, handler)` | 每次工具调用 | 日志、拦截、修改 |
| `tools`（属性） | 注册时 | 向智能体添加工具 |
| `state_schema`（属性） | 图编译时 | 用新字段扩展状态 |

用一个简单的日志中间件看看这些 hooks 如何工作：


In [ ]:
from langchain.agents.middleware import wrap_model_call, wrap_tool_call

@wrap_model_call
def inspect_model_call(request, handler):
    """See what the LLM receives on each call."""
    # request.system_message 包含完整系统提示（所有中间件都已追加）
    sys_msg = request.system_message
    if sys_msg:
        # 系统消息可能来自多个中间件的多块内容
        total_chars = sum(len(b.get("text", "")) for b in sys_msg.content_blocks if isinstance(b, dict))
        print(f"\n🧠 [Model Call] System prompt: {total_chars:,} chars across {len(sys_msg.content_blocks)} blocks")
    print(f"   Tools available: {[t.name for t in request.tools]}")
    print(f"   Messages in context: {len(request.messages)}")
    return handler(request)

@wrap_tool_call
def inspect_tool_call(request, handler):
    """See what tools are called and with what args."""
    name = request.tool_call["name"]
    args = request.tool_call["args"]
    # 截断过长参数以便显示
    display_args = {k: (v[:80] + "..." if isinstance(v, str) and len(v) > 80 else v) for k, v in args.items()}
    print(f"   🔧 {name}({display_args})")
    return handler(request)

# 把我们的检查中间件加入栈
inspectable_agent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt="You are a helpful assistant.",
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        create_summarization_middleware(model, StateBackend),
        PatchToolCallsMiddleware(),
        inspect_model_call,  # Our custom middleware
        inspect_tool_call,
    ],
).with_config({"recursion_limit": 1000})

config = {"configurable": {"thread_id": str(uuid7())}}
result = inspectable_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file called /hello.txt with 'hi'"}]
}, config=config)

print("\n---")
print(result["messages"][-1].content)

### 要点
- `create_deep_agent()` = `create_agent()` + 一套特定中间件栈
- 中间件 hooks 让你检查、修改并扩展智能体循环的任意部分
- 你可以交换、重排或替换任意中间件以定制行为


---
## 第 2 部分：构建最小化文件系统接口

Deep Agent 的 `FilesystemMiddleware` 提供 7 个工具（`ls`、`read_file`、`write_file`、`edit_file`、`glob`、`grep`、`execute`）。我们不去重写全部，而是构建一个最小 2 工具版本来理解**模式**。

**文件系统工具并不直接碰文件系统**。它们会：
1. 通过后端抽象（`BackendProtocol`）操作
2. 返回 `Command(update={"files": {...}})` 以更新 LangGraph 状态
3. 用自定义状态归约器（reducer）合并文件更新

这种模式把文件*存哪里*与智能体*如何使用*它们分离开。


In [ ]:
from datetime import datetime, timezone
from typing import Annotated, Any, NotRequired
from langchain.agents.middleware.types import AgentMiddleware, AgentState
from langchain.tools import ToolRuntime
from langchain_core.messages import ToolMessage
from langchain_core.tools import StructuredTool
from langgraph.types import Command
from langchain_core.messages import SystemMessage


# 步骤 1：定义状态归约器（reducer）
# 这是 LangGraph 合并多次工具调用文件更新的方式
def file_reducer(left: dict | None, right: dict) -> dict:
    """Merge file state updates. None values = deletions."""
    result = {**(left or {})}
    for key, value in right.items():
        if value is None:
            result.pop(key, None)  # Delete
        else:
            result[key] = value    # Create/update
    return result


# 步骤 2：定义状态 schema
class MinimalFilesystemState(AgentState):
    files: Annotated[NotRequired[dict], file_reducer]


# 步骤 3：构建中间件
class MinimalFilesystemMiddleware(AgentMiddleware):
    """A stripped-down filesystem middleware to show the pattern."""
    
    state_schema = MinimalFilesystemState
    
    def __init__(self):
        super().__init__()
        self.tools = [self._make_write_tool(), self._make_read_tool()]
    
    def _make_write_tool(self):
        def write_file(
            file_path: Annotated[str, "Absolute path for the file (e.g., /notes.md)"],
            content: Annotated[str, "Content to write"],
            runtime: ToolRuntime,
        ) -> Command:
            """Write a file to the virtual filesystem."""
            now = datetime.now(timezone.utc).isoformat()
            file_data = {
                "content": content.split("\n"),
                "created_at": now,
                "modified_at": now,
            }
            # 这就是关键模式：返回 Command 以更新状态
            return Command(update={
                "files": {file_path: file_data},
                "messages": [ToolMessage(
                    f"Wrote {len(content)} chars to {file_path}",
                    tool_call_id=runtime.tool_call_id,
                )],
            })
        
        return StructuredTool.from_function(name="write_file", func=write_file)
    
    def _make_read_tool(self):
        def read_file(
            file_path: Annotated[str, "Absolute path of the file to read"],
            runtime: ToolRuntime,
        ) -> Command:
            """Read a file from the virtual filesystem."""
            files = runtime.state.get("files", {})
            if file_path not in files:
                return Command(update={
                    "messages": [ToolMessage(
                        f"Error: {file_path} not found",
                        tool_call_id=runtime.tool_call_id,
                    )],
                })
            content = "\n".join(files[file_path]["content"])
            return Command(update={
                "messages": [ToolMessage(
                    content,
                    tool_call_id=runtime.tool_call_id,
                )],
            })
        
        return StructuredTool.from_function(name="read_file", func=read_file)
    
    def wrap_model_call(self, request, handler):
        """Tell the LLM about its filesystem capabilities."""
        instructions = """## Filesystem
You have `write_file` and `read_file` tools. Use absolute paths (e.g., `/notes.md`).
Files are stored in-memory and persist within this thread."""
        # 把我们的指令追加到已有系统消息
        existing = request.system_message
        blocks = list(existing.content_blocks) if existing else []
        blocks.append({"type": "text", "text": f"\n\n{instructions}"})
        new_sys = SystemMessage(content_blocks=blocks)
        return handler(request.override(system_message=new_sys))


print("MinimalFilesystemMiddleware defined")
print("Tools:", [t.name for t in MinimalFilesystemMiddleware().tools])

In [ ]:
# 把最小化文件系统与 create_agent 一起使用
minimal_fs_agent = create_agent(
    model,
    system_prompt="You are a helpful assistant.",
    middleware=[MinimalFilesystemMiddleware()],
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = minimal_fs_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file /demo.txt with 'Hello from minimal filesystem!' then read it back."}]
}, config=config)

print("Agent response:", result["messages"][-1].content)
print("\nFiles in state:", {k: v["content"] for k, v in result.get("files", {}).items()})

### 与真实 `FilesystemMiddleware` 的对比

我们的最小版本展示了核心模式。真实实现还增加了：

| 特性 | 我们的版本 | 真实 `FilesystemMiddleware` |
|---------|-------------|----------------------------|
| 工具 | `write_file`、`read_file` | + `ls`、`edit_file`、`glob`、`grep`、`execute` |
| 存储 | 直接访问状态 | `BackendProtocol` 抽象 |
| 分页 | 无 | 读取时的 `offset`/`limit` |
| 图片 | 无 | 多模态内容块 |
| 沙箱 | 无 | `SandboxBackendProtocol` |
| 路径校验 | 无 | 清理、虚拟路径 |

### 要点
- 文件系统工具通过 `Command(update={"files": {...}})` 修改状态，而不是直接写磁盘
- 自定义状态归约器（`file_reducer`）处理合并与删除
- `wrap_model_call` 注入指令，让 LLM 知道有哪些工具可用
- 真实实现加入后端抽象，使存储可插拔


---
## 第 3 部分：后端实现 —— 文件实际存在哪里

第 2 部分我们构建了把文件存在 LangGraph 状态里的文件系统。但这只是选项之一。Deep Agents 有三种后端实现，各自有根本不同的存储语义。

**工具不知道文件存在哪里**。它们调用 `backend.write()`，由后端决定是更新状态、写磁盘，还是调用远程 store。

### `WriteResult` 分支模式

每个后端的 `write()` 返回带三个字段的 `WriteResult`：
```python
@dataclass
class WriteResult:
    error: str | None = None        # Error message on failure
    path: str | None = None          # Path of written file
    files_update: dict | None = None  # State update dict (or None)
```

`files_update` 字段控制分支：
- **`StateBackend`**：返回 `files_update={path: file_data}` → 中间件包进 `Command(update={"files": ...})`
- **`FilesystemBackend`**：返回 `files_update=None` → 文件已在磁盘，中间件只返回字符串确认
- **`StoreBackend`**：返回 `files_update=None` → 文件已在 Store，无需状态更新

`FilesystemMiddleware._create_write_file_tool()`：
```python
res: WriteResult = backend.write(file_path, content)
if res.files_update is not None:
    # StateBackend: update LangGraph state
    return Command(update={"files": res.files_update, "messages": [...]})
# FilesystemBackend/StoreBackend: already persisted
return f"Updated file {res.path}"
```


In [ ]:
# 看看实际差异
from deepagents.backends.protocol import WriteResult

# 模拟各后端对同一次 write() 调用的返回

# StateBackend："这是数据，请放进状态"
state_result = WriteResult(
    path="/notes.md",
    files_update={"/notes.md": {"content": ["hello"], "created_at": "...", "modified_at": "..."}}
)
print(f"StateBackend:      files_update = {type(state_result.files_update).__name__} (middleware issues Command)")

# FilesystemBackend："我已经写到磁盘了"
fs_result = WriteResult(path="/notes.md", files_update=None)
print(f"FilesystemBackend: files_update = {fs_result.files_update} (middleware returns string)")

# StoreBackend："我已经放进 Store 了"
store_result = WriteResult(path="/notes.md", files_update=None)
print(f"StoreBackend:      files_update = {store_result.files_update} (middleware returns string)")

### StateBackend —— 短暂，存在 LangGraph 状态中

这是默认后端。文件存在 `runtime.state["files"]` —— 由 LangGraph 检查点管理的字典。

```python
class StateBackend(BackendProtocol):
    def __init__(self, runtime: ToolRuntime):
        self.runtime = runtime
    
    def read(self, file_path, ...):
        files = self.runtime.state.get("files", {})
        return files[file_path]  # Read directly from state
    
    def write(self, file_path, content):
        file_data = create_file_data(content)
        # Return the data so middleware can update state via Command
        return WriteResult(path=file_path, files_update={file_path: file_data})
```

**生命周期：** 文件在线程内持久（经 checkpointer），跨线程则消失。

**权衡：** 快（内存），但每个文件都会序列化进检查点。大文件会膨胀检查点体积。


In [ ]:
# StateBackend 实际效果 —— 文件存在 state["files"]
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

state_agent = create_deep_agent(
    model=model,
    backend=StateBackend(),  # Default — pass an instance
    system_prompt="You are a helpful assistant.",
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = state_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file /demo.txt with 'hello from StateBackend'"}]
}, config=config)

# 文件在结果状态中
print("Files in state:", list(result.get("files", {}).keys()))
if "/demo.txt" in result.get("files", {}):
    print("Content:", result["files"]["/demo.txt"]["content"])

### FilesystemBackend —— 真实磁盘 I/O

读写磁盘上的真实文件。关键参数是 `virtual_mode`：

- `virtual_mode=True`：路径被沙箱限制在 `root_dir` 下。智能体写 `/notes.md` → 实际写到 `{root_dir}/notes.md`。路径穿越（`..`）被阻止。
- `virtual_mode=False`（默认，已弃用）：绝对路径按原样工作。**没有安全边界。**

```python
class FilesystemBackend(BackendProtocol):
    def __init__(self, root_dir, virtual_mode=True):
        self.cwd = Path(root_dir).resolve()
        self.virtual_mode = virtual_mode
    
    def write(self, file_path, content):
        resolved = self._resolve_path(file_path)  # Sandboxing happens here
        resolved.parent.mkdir(parents=True, exist_ok=True)
        resolved.write_text(content)
        return WriteResult(path=file_path, files_update=None)  # Already on disk!
```

**与 StateBackend 的关键差异：** `files_update=None` —— 文件已持久到磁盘。中间件返回普通字符串响应，无需 `Command`。

**生命周期：** 文件永久留在磁盘。持久化不需要 checkpointer，但使用同一 `root_dir` 的所有线程共享这些文件。


In [ ]:
# FilesystemBackend 实际效果 —— 文件落到真实磁盘
import tempfile, os
from deepagents.backends import FilesystemBackend

sandbox_dir = tempfile.mkdtemp(prefix="deepagents_extra_")
print(f"Sandbox directory: {sandbox_dir}")

fs_agent = create_deep_agent(
    model=model,
    backend=FilesystemBackend(root_dir=sandbox_dir, virtual_mode=True),
    system_prompt="You are a helpful assistant.",
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = fs_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file /demo.txt with 'hello from FilesystemBackend'"}]
}, config=config)

# 状态中没有文件 —— 它们在磁盘上！
print(f"Files in state: {list(result.get('files', {}).keys())}")

# 但它们确实在磁盘上
disk_path = os.path.join(sandbox_dir, "demo.txt")
if os.path.exists(disk_path):
    print(f"File on disk: {open(disk_path).read()}")

# 清理
import shutil
shutil.rmtree(sandbox_dir, ignore_errors=True)

### StoreBackend —— 持久、跨线程存储

使用 LangGraph 的 `BaseStore`（键值存储）。文件跨所有线程持久 —— 在线程 A 写入，在线程 B 读取。

```python
class StoreBackend(BackendProtocol):
    def __init__(self, runtime: ToolRuntime, *, namespace=None):
        self.runtime = runtime
    
    def write(self, file_path, content):
        store = self.runtime.store          # LangGraph provides this
        namespace = self._get_namespace()    # e.g., ("filesystem",)
        file_data = create_file_data(content)
        store.put(namespace, file_path, file_data)  # Persisted immediately
        return WriteResult(path=file_path, files_update=None)  # Already in store!
```

**命名空间作用域**控制隔离：
- `("filesystem",)` —— 所有线程 / 用户共享文件
- `(user_id, "filesystem")` —— 按用户隔离
- `(assistant_id, "filesystem")` —— 按助手隔离

**生命周期：** 只要 Store 存在，文件就持久。在 LangGraph Platform 上意味着可跨部署存活。


In [ ]:
# StoreBackend 实际效果 —— 文件跨线程持久
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend

store = InMemoryStore()

store_agent = create_deep_agent(
    model=model,
    backend=StoreBackend(namespace=lambda rt: ("filesystem",)),
    store=store,
    system_prompt="You are a helpful assistant.",
    checkpointer=MemorySaver(),
)

# 线程 1：写入文件
config_t1 = {"configurable": {"thread_id": str(uuid7())}}
result = store_agent.invoke({
    "messages": [{"role": "user", "content": "Write a file /shared_note.txt with 'This persists across threads!'"}]
}, config=config_t1)
print("Thread 1:", result["messages"][-1].content)

# 线程 2：读取它（不同线程！）
config_t2 = {"configurable": {"thread_id": str(uuid7())}}
result = store_agent.invoke({
    "messages": [{"role": "user", "content": "Read the file /shared_note.txt"}]
}, config=config_t2)
print("Thread 2:", result["messages"][-1].content)

### 后端对比

| | StateBackend | FilesystemBackend | StoreBackend |
|---|---|---|---|
| **存储** | LangGraph 状态（检查点） | 真实文件系统 | LangGraph Store（键值） |
| **持久性** | 线程内 | 磁盘永久 | 跨线程、跨会话 |
| **`files_update`** | `{path: data}` → `Command` | `None` → 字符串 | `None` → 字符串 |
| **状态膨胀** | 是（文件在检查点中） | 否 | 否 |
| **适用场景** | 草稿空间、notebook | CLI 工具、本地开发 | LangGraph Platform、生产 |
| **需要 `store=`** | 否 | 否 | 是（`InMemoryStore` 或平台 store） |

生产中通常用 `CompositeBackend` 混合它们 —— 经 `StateBackend` 做短暂草稿，经 `StoreBackend` 做持久记忆：

```python
composite_backend = CompositeBackend(
    default=StateBackend(),           # Scratch files go to state
    routes={
        "/memories/": StoreBackend(),  # Memories persist in Store
    },
)
```

### 要点
- `WriteResult.files_update` 字段控制中间件是发 `Command` 还是返回字符串
- `StateBackend` 返回用于状态更新的数据；`FilesystemBackend` 与 `StoreBackend` 直接持久并返回 `None`
- 三种后端都实现同一 `BackendProtocol` —— 工具不知道用的是哪一种
- `CompositeBackend` 把不同路径路由到不同后端，实现混合存储


---
## 第 4 部分：子智能体的自定义任务定义

默认情况下，`SubAgentMiddleware` 创建带两个参数的 `task()` 工具：
- `description`（str）—— 描述要做什么的原始字符串
- `subagent_type`（str）—— 调用哪个子智能体

子智能体收到 `HumanMessage(content=description)` —— 只是扁平字符串。

这能工作，但有局限：
- 无结构 —— LLM 必须把一切塞进散文
- 无类型安全 —— 对传入内容没有校验
- 无选择性上下文 —— 子智能体拿到所有未排除的状态

下面构建能修复这些问题的自定义任务工具。


In [ ]:
# 默认 task 工具流程（简化）：
# 1. 主智能体调用：task(description="Research LangGraph", subagent_type="researcher")
# 2. SubAgentMiddleware：
# a. 过滤状态：移除内部键（messages、todos 等），让子智能体获得干净上下文
# b. 设置 messages: [HumanMessage("Research LangGraph")]  <-- 只是原始字符串
# c. 调用子智能体
# d. 返回：Command(update={"messages": [ToolMessage(result)]})
#
# 局限："description" 是原始字符串。LLM 必须把
# 一切 —— 主题、深度、格式、约束 —— 塞进散文。
# 我们用结构化输入来修复。

In [ ]:
# 自定义任务定义 1：结构化输入
#
# 不用原始字符串，而为任务输入定义 Pydantic 模型。
# 这给编排智能体一个清晰可填的 schema。

from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage


class ResearchTask(BaseModel):
    """A structured research task definition."""
    topic: str = Field(description="The topic to research")
    depth: str = Field(description="'shallow' (quick overview), 'medium' (key details), or 'deep' (comprehensive)")
    output_format: str = Field(description="'bullet_points', 'narrative', or 'structured_report'")
    max_sources: int = Field(default=3, description="Maximum number of sources to consult")
    focus_areas: list[str] = Field(default_factory=list, description="Specific aspects to focus on")


def build_structured_task_tool(subagent_graph):
    """Build a task tool with structured input instead of a raw string."""
    
    def research(
        topic: Annotated[str, "The topic to research"],
        depth: Annotated[str, "'shallow', 'medium', or 'deep'"],
        output_format: Annotated[str, "'bullet_points', 'narrative', or 'structured_report'"],
        max_sources: Annotated[int, "Maximum sources to consult"] = 3,
        focus_areas: Annotated[list[str], "Specific aspects to focus on"] = [],
        runtime: ToolRuntime = None,
    ) -> Command:
        """Delegate a structured research task to the research subagent."""
        # 从类型化字段构建结构良好的提示
        focus_str = "\n".join(f"  - {area}" for area in focus_areas) if focus_areas else "  (none specified)"
        
        prompt = f"""## Research Task

**Topic:** {topic}
**Depth:** {depth}
**Max Sources:** {max_sources}

**Focus Areas:**
{focus_str}

**Output Format:** {output_format}

Conduct the research and return your findings in the specified format."""
        
        # 从干净状态开始：只有结构化提示与共享文件
        subagent_state = {
            "messages": [HumanMessage(content=prompt)],
        }
        # 若父状态中有文件则可选传入
        files = runtime.state.get("files")
        if files:
            subagent_state["files"] = files
        
        result = subagent_graph.invoke(subagent_state)
        
        return Command(update={
            "messages": [ToolMessage(
                result["messages"][-1].text.rstrip(),
                tool_call_id=runtime.tool_call_id,
            )],
        })
    
    return StructuredTool.from_function(
        name="research",
        func=research,
        description="Delegate a structured research task to a specialist subagent.",
    )


print("Structured task tool builder defined")

In [ ]:
# 创建研究子智能体与编排器

# 子智能体：带搜索能力的简单 create_agent
research_subagent = create_agent(
    model,
    tools=[tavily_search],
    system_prompt="You are a research specialist. Follow the task instructions precisely.",
    middleware=[
        PatchToolCallsMiddleware(),
    ],
)

# 构建结构化任务工具
structured_research_tool = build_structured_task_tool(research_subagent)

# 编排器：使用结构化任务工具
orchestrator = create_agent(
    model,
    tools=[structured_research_tool],
    system_prompt="""You are a research coordinator. When asked to research something:
1. Use the `research` tool to delegate to a specialist
2. Always specify depth, output_format, and relevant focus_areas
3. Summarize the results for the user""",
    middleware=[
        TodoListMiddleware(),
        FilesystemMiddleware(backend=StateBackend),
        PatchToolCallsMiddleware(),
    ],
).with_config({"recursion_limit": 100})

print("Orchestrator ready with structured research tool")

In [ ]:
# 测试：编排器现在传入结构化字段，而非原始字符串
config = {"configurable": {"thread_id": str(uuid7())}}

result = orchestrator.invoke({
    "messages": [{"role": "user", "content": "Research what's new with LangGraph in 2025. Focus on the agent framework and middleware system. Give me bullet points."}]
}, config=config)

print(result["messages"][-1].content[:2000])

### 要点
- 默认 `task()` 工具把原始字符串作为 `HumanMessage` 传入 —— 你可用结构化 Pydantic 字段替换
- 自定义任务工具让你完全控制子智能体收到的状态


---
## 第 5 部分：把 StateGraph 工作流当作子智能体

子智能体不必是智能体（LLM 循环）。你可以接入任何输出含 `messages` 的 `Runnable` —— 包括确定性的 `StateGraph` 工作流。

当你有一条**固定流水线**（收集 → 分析 → 格式化）、不该交给 LLM 编排时，这很强大。


In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# 定义工作流状态 —— 对 CompiledSubAgent 必须包含 `messages`
class ResearchWorkflowState(TypedDict):
    messages: list          # Required: CompiledSubAgent reads the last message
    topic: str              # Extracted from the incoming HumanMessage
    raw_data: str           # Output of the gather step
    analysis: str           # Output of the analyze step


# 步骤 1：提取主题并收集数据
def gather_data(state: ResearchWorkflowState):
    """Search for information on the topic."""
    # task 工具发送 HumanMessage(description) —— 提取主题
    topic = state["messages"][-1].content
    
    # 直接使用 tavily（不是作为工具 —— 这是确定性步骤）
    results = tavily_client.search(topic, max_results=3)
    raw = "\n\n".join(
        f"Source: {r['url']}\n{r['content']}" 
        for r in results["results"]
    )
    return {"topic": topic, "raw_data": raw}


# 步骤 2：对原始数据做 LLM 分析
def analyze_data(state: ResearchWorkflowState):
    """Have the LLM analyze and synthesize the gathered data."""
    response = model.invoke([
        {"role": "system", "content": "Analyze the following research data. Identify key themes, facts, and insights. Be concise."},
        {"role": "user", "content": f"Topic: {state['topic']}\n\nData:\n{state['raw_data']}"}
    ])
    return {"analysis": response.content}


# 步骤 3：格式化最终报告并为父智能体设置 messages
def format_report(state: ResearchWorkflowState):
    """Format into a final report and return via messages."""
    response = model.invoke([
        {"role": "system", "content": "Write a concise research summary with bullet points based on this analysis."},
        {"role": "user", "content": state["analysis"]}
    ])
    # 最后一条消息会返回给父智能体
    return {"messages": [AIMessage(content=response.content)]}


# 构建工作流图
workflow = StateGraph(ResearchWorkflowState)
workflow.add_node("gather", gather_data)
workflow.add_node("analyze", analyze_data)
workflow.add_node("format", format_report)
workflow.add_edge(START, "gather")
workflow.add_edge("gather", "analyze")
workflow.add_edge("analyze", "format")
workflow.add_edge("format", END)

compiled_workflow = workflow.compile()
print("Research workflow compiled: gather → analyze → format")

In [ ]:
# 先单独测试工作流
standalone_result = compiled_workflow.invoke({
    "messages": [HumanMessage(content="Latest developments in AI agents 2025")]
})

print("Workflow output (last message):")
print(standalone_result["messages"][-1].content[:1000])

In [ ]:
# 现在把它作为 CompiledSubAgent 接入
from deepagents import create_deep_agent

agent_with_workflow = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a research coordinator. Use the research-pipeline for structured research.",
    subagents=[
        {
            # CompiledSubAgent：状态中含 'messages' 的任意 Runnable
            "name": "research-pipeline",
            "description": "A deterministic research pipeline that gathers data, analyzes it, and produces a structured report. Use this for any research request.",
            "runnable": compiled_workflow,
        },
    ],
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_workflow.invoke({
    "messages": [{"role": "user", "content": "Research the current state of LangGraph's middleware system"}]
}, config=config)

print(result["messages"][-1].content[:2000])

### 智能体子智能体 vs 工作流子智能体

| | 智能体子智能体（默认） | 工作流子智能体 |
|---|---|---|
| **执行** | LLM 决定步骤 | 固定流水线 |
| **灵活性** | 可临场调整 | 确定性 |
| **成本** | 更多 tokens（LLM 推理） | 更少 tokens（结构化步骤） |
| **调试** | 更难（LLM 选择会变） | 更容易（固定路径） |
| **何时使用** | 开放式任务 | 定义清晰的流水线 |

`CompiledSubAgent` 契约很简单：你的 `Runnable` 输出状态中必须有 `messages`。最后一条消息会作为 `ToolMessage` 返回给父智能体。

### 要点
- 任何输出状态含 `messages` 的 `Runnable` 都可通过 `CompiledSubAgent` 成为子智能体
- 确定性工作流比智能体子智能体更便宜、更可预测
- 两者可混合：开放式任务用智能体子智能体，流水线用工作流子智能体


---
## 第 6 部分：用于工具的 MCP 适配器

[Model Context Protocol (MCP)](https://modelcontextprotocol.io/) 提供从外部服务器暴露工具的标准方式。关键洞见是：**MCP 工具会变成普通的 LangChain `BaseTool` 对象**，因此可与任何其他工具互换。

### MCP 工具适配器如何工作

```
┌─────────────┐     ┌───────────────┐     ┌─────────────┐
│  MCP Server  │────▶│  MCP Adapter   │────▶│  BaseTool    │
│ (external)   │     │ (langchain-mcp)│     │ (standard)   │
└─────────────┘     └───────────────┘     └─────────────┘
                                                │
                    ┌───────────────────────────┘
                    ▼
        create_agent(tools=[mcp_tool, regular_tool])
```

1. **编写**带 `FastMCP` 的 MCP 服务器（或使用现成的）
2. **连接** —— 经 stdio 或 HTTP/SSE，使用 `langchain-mcp-adapters`
3. **加载**工具 —— 用 `load_mcp_tools()`，它们变成标准 `BaseTool`
4. **使用** —— 像任何其他 LangChain 工具一样

下面用带邮件工具的 MCP 服务器构建真实例子。


In [ ]:
# 我们的 MCP 服务器在 mcp/email_tools.py —— 带 3 个工具的 FastMCP 服务器：
# write_email、check_calendar_availability、schedule_meeting
# 经 stdio 传输运行（作为子进程启动）。
#
# 连接它并加载工具。

import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

mcp_server_path = Path().resolve().parent.parent / "mcp" / "email_tools.py"
server_params = StdioServerParameters(
    command=sys.executable,              # Same Python interpreter (respects venv)
    args=["-u", str(mcp_server_path)],   # -u = unbuffered stdout (required for stdio)
)

async def demo_mcp_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # load_mcp_tools 把 MCP 工具转换成标准 LangChain BaseTool 对象
            tools = await load_mcp_tools(session)
            
            print(f"Loaded {len(tools)} MCP tools:\n")
            for tool in tools:
                print(f"  {tool.name}: {tool.description}")
                print(f"    Type: {type(tool).__name__}")
                print()
            
            # 直接调用一个以证明可用
            result = await tools[0].ainvoke({"to": "alice@example.com", "subject": "Hello", "content": "Hi Alice!"})
            print(f"Direct tool call result: {result}")

await demo_mcp_tools()

In [ ]:
# 把 MCP 工具与 create_agent 一起用 —— 它们是标准 BaseTool

async def demo_agent_with_mcp():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            
            # 把 MCP 工具与普通工具混合
            all_tools = [tavily_search, *tools]
            
            agent = create_agent(
                model,
                tools=all_tools,
                system_prompt="You are a helpful assistant with email and calendar capabilities.",
                middleware=[PatchToolCallsMiddleware()],
            )
            
            config = {"configurable": {"thread_id": str(uuid7())}}
            result = await agent.ainvoke({
                "messages": [{"role": "user", "content": "Check my calendar availability for Monday and send an email to bob@example.com letting him know when I'm free."}]
            }, config=config)
            
            print(result["messages"][-1].content)

await demo_agent_with_mcp()

### 何时 MCP 会改变你的调用模式

MCP 工具是**远程过程调用（RPC）**。这有实际影响：

| 方面 | 原生 `@tool` | MCP 工具 |
|--------|----------------|----------|
| 延迟 | 进程内 | 网络往返（子进程或 HTTP） |
| 状态访问 | 可用 `ToolRuntime` | 无状态访问（无状态 RPC） |
| 错误处理 | Python 异常 | MCP 错误协议 |
| 生命周期 | 与智能体同生 | 服务器进程必须在运行 |

最重要的差异：**MCP 工具无法访问 `ToolRuntime`**。它们不能读取智能体状态、返回 `Command` 对象，或直接修改状态。它们是纯输入 → 输出函数。

这意味着你不能用 MCP 文件系统工具替换 `FilesystemMiddleware` 工具并得到同样的状态管理。MCP 工具最适合**外部能力**（邮件、日历、数据库、API），而非与状态耦合的操作。

### 要点
- MCP 服务器经 `FastMCP` 暴露工具 —— 写一次，任意 MCP 客户端可用
- `load_mcp_tools()` 把 MCP 工具转换成标准 `BaseTool`
- 可自由混合 MCP 与原生工具 —— 智能体看不出差别
- MCP 工具是无状态 RPC —— 不能访问智能体状态或返回 `Command`


---
## 第 7 部分：沙箱用法

当后端实现 `SandboxBackendProtocol` 时，Deep Agents 可以执行 shell 命令。这会给智能体增加一个 `execute` 工具。

### 沙箱后端层级

```
BackendProtocol               <- File operations only (ls, read, write, edit, glob, grep)
  └── SandboxBackendProtocol  <- + execute(command, timeout) + id property
        └── BaseSandbox       <- Implements ALL file ops as shell commands via execute()
              └── LangSmithSandbox   <- Wraps langsmith.sandbox.SandboxClient
              └── YourOwnSandbox     <- Just implement execute()!
```

`BaseSandbox` 通过 `execute()` 用 shell 命令实现**所有**文件操作（`read`、`write`、`edit`、`ls`、`glob`、`grep`）。因此要构建自己的沙箱后端，你只需实现**一个方法**：`execute()`。

### LangSmith Sandbox SDK

```python
from langsmith.sandbox import SandboxClient

client = SandboxClient()  # Uses LANGSMITH_API_KEY
client.create_template(name="my-sandbox", image="python:3.12-slim")

with client.sandbox(template_name="my-sandbox") as sb:
    sb.write("/app/script.py", "print('hello')")
    result = sb.run("python /app/script.py")
    print(result.stdout)  # "hello\n"
```

Deep Agents 用 `LangSmithSandbox(sb)` 包装它，暴露为 `SandboxBackendProtocol`。


In [ ]:
# 控制沙箱能力的协议
from deepagents.backends.protocol import BackendProtocol, SandboxBackendProtocol, ExecuteResponse

# BackendProtocol 方法（文件操作）：
backend_methods = [m for m in dir(BackendProtocol) if not m.startswith('_') and not m.startswith('a')]
print("BackendProtocol methods:", backend_methods)

# SandboxBackendProtocol 增加：
sandbox_only = [m for m in dir(SandboxBackendProtocol) if not m.startswith('_') and not m.startswith('a') and m not in dir(BackendProtocol)]
print("SandboxBackendProtocol adds:", sandbox_only)

print(f"\nExecuteResponse fields: output (str), exit_code (int|None), truncated (bool)")

In [ ]:
# （可选）运行示例：LangSmith Sandbox
# 需要支持沙箱的套餐所对应的 LANGSMITH_API_KEY。
# 若无权限可跳过本单元格 —— 下面概念部分已覆盖该模式。

import os

if os.environ.get("LANGSMITH_API_KEY"):
    try:
        from langsmith.sandbox import SandboxClient
        from deepagents.backends.langsmith import LangSmithSandbox

        client = SandboxClient()

        # 创建模板（只需一次 —— 再调用也安全）
        try:
            client.create_template(name="deepagents-extra", image="python:3.12-slim")
            print("Created template 'deepagents-extra'")
        except Exception:
            print("Template 'deepagents-extra' already exists")

        # 启动沙箱并运行命令
        with client.sandbox(template_name="deepagents-extra") as sb:
            # 直接使用 SDK
            result = sb.run("echo 'Hello from sandbox!' && python3 --version")
            print(f"\nDirect execution:")
            print(f"  stdout: {result.stdout.strip()}")
            print(f"  success: {result.success}")

            # 文件操作
            sb.write("/app/hello.py", "print('Hello from a sandboxed Python script!')")
            result = sb.run("python3 /app/hello.py")
            print(f"\nScript execution:")
            print(f"  stdout: {result.stdout.strip()}")

            # 包装为 Deep Agents 后端并与 create_deep_agent 一起用
            backend = LangSmithSandbox(sb)
            print(f"\nSandbox ID: {backend.id}")
            print(f"Backend type: {type(backend).__name__}")
            print(f"Is SandboxBackendProtocol: {isinstance(backend, SandboxBackendProtocol)}")

            sandbox_agent = create_deep_agent(
                model=model,
                backend=backend,
                system_prompt="You are a coding assistant. Write and run code in the sandbox.",
            )

            config = {"configurable": {"thread_id": str(uuid7())}}
            result = sandbox_agent.invoke({
                "messages": [{"role": "user", "content": "Write a Python one-liner to /app/fib.py that prints the first 10 fibonacci numbers, then run it."}]
            }, config=config)
            print(f"\nAgent response: {result['messages'][-1].content}")

    except Exception as e:
        print(f"Sandbox error: {e}")
        print("Sandboxes require a LangSmith plan that supports them.")
else:
    print("LANGSMITH_API_KEY not set — skipping LangSmith sandbox demo.")
    print("Set it in your .env to run this cell.")

In [ ]:
# 构建自己的沙箱后端
#
# BaseSandbox 经 execute() 处理全部文件操作。
# 你只需实现：execute()、id、upload_files()、download_files()
#
# 模式如下（简化自 langchain-ai/openshell-deepagent）：

from deepagents.backends.sandbox import BaseSandbox
from deepagents.backends.protocol import ExecuteResponse, FileUploadResponse, FileDownloadResponse
import subprocess

class LocalDockerSandbox(BaseSandbox):
    """Example: run commands in a local Docker container.
    
    In production you'd use a real sandbox service (LangSmith, OpenShell, Modal, etc.)
    This just demonstrates the pattern.
    """
    
    def __init__(self, container_id: str):
        self._container_id = container_id
    
    @property
    def id(self) -> str:
        return self._container_id
    
    def execute(self, command: str, *, timeout: int | None = None) -> ExecuteResponse:
        """Run a command inside the container — this is the only method you must implement."""
        try:
            result = subprocess.run(
                ["docker", "exec", self._container_id, "bash", "-c", command],
                capture_output=True, text=True,
                timeout=timeout or 60,
            )
            output = result.stdout
            if result.stderr:
                output = f"{output}\n{result.stderr}" if output else result.stderr
            return ExecuteResponse(output=output, exit_code=result.returncode)
        except subprocess.TimeoutExpired:
            return ExecuteResponse(output="Command timed out", exit_code=1)
    
    # upload_files 与 download_files 在 BaseSandbox 中有默认实现
    # 它们用带 base64 编码的 execute()。仅当你的沙箱
    # 有更高效的原生文件传输机制时才覆盖。

print("LocalDockerSandbox defined — demonstrates the BaseSandbox pattern")
print()
print("What BaseSandbox gives you for free (all via execute()):")
print("  ls, read, write, edit, glob, grep")
print()
print("What you implement:")
print("  execute() — required")
print("  id — required (property)")
print("  upload_files() — optional (default uses execute + base64)")
print("  download_files() — optional (default uses execute + base64)")

### 把沙箱后端与 `create_deep_agent` 一起使用

**模式 1：直接使用沙箱后端**
```python
agent = create_deep_agent(
    model=model,
    backend=LocalDockerSandbox("my-container-id"),
)
```

**模式 2：CompositeBackend —— 代码用沙箱，记忆用文件系统**
```python
from deepagents.backends import CompositeBackend, FilesystemBackend

def create_backend(runtime):
    return CompositeBackend(
        default=LocalDockerSandbox("my-container-id"),
        routes={
            "/memories/": FilesystemBackend(root_dir="./data", virtual_mode=True),
        },
    )

agent = create_deep_agent(model=model, backend=create_backend)
```

因为后端实现了 `SandboxBackendProtocol`，智能体会自动获得 `execute` 工具。


### 要点
- 要构建自己的沙箱后端，继承 `BaseSandbox` 并实现 `execute()` —— 就这些
- `BaseSandbox` 经 shell 命令从 `execute()` 派生全部文件操作（`ls`、`read`、`write` 等）
- 用 `CompositeBackend` 混合沙箱（代码执行）与 `FilesystemBackend`（持久记忆 / skills）
- `LangSmithSandbox` 是生产就绪实现；另一个例子见 [openshell-deepagent](https://github.com/langchain-ai/openshell-deepagent)


---
## 总结

| 部分 | 你学到了什么 | 关键模式 |
|------|-----------------|-------------|
| **1. 拆解** | `create_deep_agent` = `create_agent` + 中间件 | 中间件栈组合 |
| **2. 文件系统** | 工具返回 `Command(update={...})` 修改状态 | 状态归约器 + Command 模式 |
| **3. 后端** | `WriteResult.files_update` 控制状态 vs 直接持久 | 后端抽象、`CompositeBackend` |
| **4. 任务定义** | 用结构化 Pydantic 输入替换原始字符串任务 | 每个子智能体自定义 `StructuredTool` |
| **5. 工作流子智能体** | 任何含 `messages` 的 `Runnable` 都可作子智能体 | `CompiledSubAgent` 契约 |
| **6. MCP 适配器** | MCP 工具变成标准 `BaseTool` | 适配器模式、无状态 RPC |
| **7. 沙箱** | `SandboxBackendProtocol` 控制 `execute` 工具 | 协议扩展模式 |

### 接下来可以去哪

- **构建自己的中间件** —— 理解 hooks 后，可做日志、缓存、限流或自定义行为
- **自定义后端** —— 为自己的存储（S3、数据库等）实现 `BackendProtocol`
- **用 LangGraph Platform 部署** —— 用 `langgraph dev` 带着 Studio UI 运行智能体
- **评估** —— 用 LangSmith 通过数据集与评估器衡量智能体质量
